<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgba(62, 128, 234, 1);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ crime buffer visualization ★ </h3>
  <p> This notebook is an experimental notebook for visualizing crime buffer segment scoring.</p>
</div>

In [1]:
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import plotly.express as px
import geopandas as gpd
import math
import plotly.graph_objects as go 
import geopandas as gpd
import seaborn as sns
from shapely import wkt
import folium
import ast
import branca
from shapely import LineString, MultiLineString
import matplotlib.patches as patches
import matplotlib.colors as mcolors
import os
import pathlib
import nbformat
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable, get_cmap
from plotly.subplots import make_subplots
import webcolors as wc
import osmnx as ox
import os, sys
import builtins
import networkx as nx
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely import wkt
from shapely.geometry import box
import folium
import warnings
warnings.filterwarnings("ignore")
import csv
from IPython.display import display
from html2image import Html2Image
from PIL import Image, ImageDraw, ImageFont
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import time
import tempfile
import math
import folium
from shapely.geometry import LineString
from shapely.ops import unary_union
from typing import Callable, Dict, Iterable, Optional, Set, Tuple

# ignore warnings globally
warnings.filterwarnings("ignore")

# columns to be plotted
d = "buffer_blended_day"
n = "buffer_blended_night"


<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgba(62, 128, 234, 1);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ csvs ★ </h3>
  <p> read in the edges with scores csvs here.</p>
</div>

In [2]:
print(f'Current Directory: {os.getcwd()}')
data_dir = Path('data/scores')  

# match filename pattern
pattern = re.compile(
    r"edges_with_safety_score_(street|point)_buffer_(\d+)m\.csv$"
)

matched = []
for p in data_dir.glob('*.csv'):
    m = pattern.match(p.name)
    if m:
        sigma_val = float(m.group(2))
        matched.append((sigma_val, p))

# sort by the numeric sigma
matched.sort(key=lambda tup: tup[0])

# unpack
csvs = [p for _, p in matched]

# add direct mapping file
direct_file = data_dir / "edges_with_safety_score_direct_mapping.csv"
if direct_file.exists():
    csvs = [direct_file] + csvs

print(csvs)
num_times = len(csvs) 

Current Directory: /Users/alice/Documents/GitHub/STARSWalkability/crime
[PosixPath('data/scores/edges_with_safety_score_direct_mapping.csv'), PosixPath('data/scores/edges_with_safety_score_point_buffer_10m.csv'), PosixPath('data/scores/edges_with_safety_score_street_buffer_10m.csv'), PosixPath('data/scores/edges_with_safety_score_street_buffer_15m.csv'), PosixPath('data/scores/edges_with_safety_score_point_buffer_15m.csv'), PosixPath('data/scores/edges_with_safety_score_point_buffer_20m.csv'), PosixPath('data/scores/edges_with_safety_score_street_buffer_20m.csv'), PosixPath('data/scores/edges_with_safety_score_street_buffer_25m.csv'), PosixPath('data/scores/edges_with_safety_score_point_buffer_25m.csv'), PosixPath('data/scores/edges_with_safety_score_point_buffer_40m.csv'), PosixPath('data/scores/edges_with_safety_score_point_buffer_50m.csv')]


In [3]:
# extract a plain name from a filename
def label_from_path(path: Path) -> str:
    name = path.name
    if "direct_mapping" in name:
        return "Direct Mapping"

    # buffer
    m = re.match(r"edges_with_safety_score_(street|point)_buffer_(\d+)m\.csv$", name)
    if m:
        buf_type = m.group(1).capitalize()
        dist = m.group(2)
        return f"{buf_type} Buffer {dist}m"
    
    return name

<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgba(62, 128, 234, 1);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ histograms ★ </h3>
  <p> plotting different histograms for various buffer sizes.</p>
</div>

In [5]:
out_dir = Path("../crime/data/graphs/hist")
out_dir.mkdir(parents=True, exist_ok=True)

 # lowercase, replace spaces/unsafe chars with underscores
def safe_stem(s: str) -> str:
    return re.sub(r'[^a-z0-9._-]+', '_', s.lower())

for i in range(num_times):
    buffer = label_from_path(csvs[i])

    # put vals into df
    scores = pd.read_csv(csvs[i])
    scores_day = scores[d].dropna()
    scores_night = scores[n].dropna()
    fig, axes = plt.subplots(
        nrows=1, ncols=2,
        figsize=(12, 6),
        constrained_layout=True
    )

    # plot day
    ax=axes[0]
    scores_day.hist(ax=ax,bins=30)
    ax.set_title(f"Buffer-based Safety Score Distribution - Day ({buffer})")
    ax.set_xlabel("Buffer-based Safety Score")
    ax.set_ylabel("Frequency")

    # plot night
    ax=axes[1]
    scores_night.hist(ax=ax,bins=30)
    ax.set_title(f"Buffer-based Safety Score Distribution - Night ({buffer})")
    ax.set_xlabel("Buffer-based Safety Score")
    ax.set_ylabel("Frequency")

    # save image
    stem = safe_stem(buffer)
    out_path = out_dir / f"{stem}_day_night.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig) 

<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgba(62, 128, 234, 1);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ heatmaps ★ </h3>
  <p> segment heatmaps for various buffer sizes and score types.</p>
</div>

In [13]:
# read in df, convert to gdf, get geometry column
def load_gdf_from_csv(path, wkt_col="geometry", crs="EPSG:4326"):
    df = pd.read_csv(path)
    if wkt_col not in df.columns:
        raise ValueError(f"'{wkt_col}' column not found in {path}")
    geom = gpd.GeoSeries.from_wkt(df[wkt_col])
    gdf = gpd.GeoDataFrame(df, geometry=geom, crs=crs)
    return gdf

# plot day/night graphs next to each other, saves as png
def plot_gdf_day_night_pair(
    csv_path: Path,
    day_col: str,
    night_col: str,
    title_label: str,
    wkt_col: str = "geometry",
    crs: str = "EPSG:4326",
    cmap: str = "Reds_r",
    figsize=(12, 6),
    share_extent: bool = True,
    save_dir: Path = Path("../crime/data/graphs/gdfs"),
    dpi: int = 200
):
    
    save_dir.mkdir(parents=True, exist_ok=True)
    gdf = load_gdf_from_csv(csv_path, wkt_col=wkt_col, crs=crs)

    for col in (day_col, night_col):
        if col not in gdf.columns:
            raise ValueError(f"Column '{col}' not found in {csv_path.name}")

    # pairwise-global vmin/vmax
    all_vals = pd.concat([gdf[day_col], gdf[night_col]], ignore_index=True).dropna()
    if all_vals.empty:
        raise ValueError(f"No numeric values in {day_col}/{night_col} for {csv_path.name}")
    vmin, vmax = float(all_vals.min()), float(all_vals.max())

    # extent
    xlim = (gdf.total_bounds[0], gdf.total_bounds[2])
    ylim = (gdf.total_bounds[1], gdf.total_bounds[3])

    # figure & axes
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=figsize, constrained_layout=True)

    # left: day
    ax = axes[0]
    gdf.plot(column=day_col, ax=ax, legend=False, vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_title(f"Day ({title_label})", fontsize=20)
    ax.set_axis_off()
    if share_extent:
        ax.set_xlim(xlim); ax.set_ylim(ylim)

    # right: night
    ax = axes[1]
    gdf.plot(column=night_col, ax=ax, legend=False, vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_title(f"Night ({title_label})", fontsize=20)
    ax.set_axis_off()
    if share_extent:
        ax.set_xlim(xlim); ax.set_ylim(ylim)

    # shared colorbar
    norm = Normalize(vmin=vmin, vmax=vmax)
    sm = ScalarMappable(norm=norm, cmap=get_cmap(cmap))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes, orientation="vertical", fraction=0.02, pad=0.02)
    cbar.set_label("Safety Score", fontsize=20, labelpad=10)
    cbar.ax.tick_params(labelsize=15)

    # overall title
    fig.suptitle("Buffer-based Safety Score Maps", y=1.05, fontsize=22)

    # save
    stem = safe_stem(title_label)
    out_path = save_dir / f"{stem}_gdf_day_night.png"
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    print(f"[saved] {out_path}")
    plt.close(fig)

# generate for each buffer type/sze
out_dir_gdf = Path("../crime/data/graphs/gdfs")
for i in range(len(csvs)):
    buffer_label = label_from_path(csvs[i])
    plot_gdf_day_night_pair(
        csv_path=csvs[i],
        day_col=d,
        night_col=n,
        title_label=buffer_label,
        wkt_col="geometry",
        crs="EPSG:4326",
        cmap="Reds_r",
        figsize=(14, 6),
        share_extent=True,
        save_dir=out_dir_gdf,
        dpi=200
    )

[saved] ../crime/data/graphs/gdfs/direct_mapping_gdf_day_night.png
[saved] ../crime/data/graphs/gdfs/point_buffer_10m_gdf_day_night.png
[saved] ../crime/data/graphs/gdfs/street_buffer_10m_gdf_day_night.png
[saved] ../crime/data/graphs/gdfs/street_buffer_15m_gdf_day_night.png
[saved] ../crime/data/graphs/gdfs/point_buffer_15m_gdf_day_night.png
[saved] ../crime/data/graphs/gdfs/point_buffer_20m_gdf_day_night.png
[saved] ../crime/data/graphs/gdfs/street_buffer_20m_gdf_day_night.png
[saved] ../crime/data/graphs/gdfs/street_buffer_25m_gdf_day_night.png
[saved] ../crime/data/graphs/gdfs/point_buffer_25m_gdf_day_night.png
[saved] ../crime/data/graphs/gdfs/point_buffer_40m_gdf_day_night.png
[saved] ../crime/data/graphs/gdfs/point_buffer_50m_gdf_day_night.png


<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgba(62, 128, 234, 1);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ colors and df ★ </h3>
  <p> makes a df with all the datasets combined, defines colors for buffer sizes</p>
</div>

In [15]:
# pass in csvs, extract score col from each csv, combine into one long df
def build_long(score_col, csvs, idcol=None):
    frames = []
    usecols = [score_col] if idcol is None else [idcol, score_col]

    for label, path in csvs.items():
        df = pd.read_csv(path, usecols=usecols)
        df = df.rename(columns={score_col: 'value'})
        df['dataset'] = label
        frames.append(df[['value', 'dataset']])

    return pd.concat(frames, ignore_index=True)

In [16]:
# choose colors for day by buffer size
DAY_BASE = {
    '10': "#ff0000",
    '15': "#ff7300",
    '20': "#f6ff00",
    '25': "#3191fe",
}

# choose colors for night by buffer size
NIGHT_BASE = {
    '10': "#00AEFF",
    '15': "#0d45ff",
    '20': "#6c03ff",
    '25': "#ff00ff",
}

In [17]:
_DEFAULT_BASE = {
    '25': '#17becf',
    '20': '#2ca02c',
    '15': '#ff7f0e',
    '10': '#d62728',
}

def _hex2rgb(h: str):
    """#RRGGBB → (r,g,b). Accepts #RGB too."""
    h = h.lstrip('#')
    if len(h) == 3:
        h = ''.join(ch * 2 for ch in h)
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))


def _rgb2hex(rgb):
    r, g, b = rgb
    return f"#{r:02x}{g:02x}{b:02x}"


def _mix(c1, c2, t: float):
    r1, g1, b1 = _hex2rgb(c1)
    r2, g2, b2 = _hex2rgb(c2)
    return _rgb2hex((
        int(round(r1 + (r2 - r1) * t)),
        int(round(g1 + (g2 - g1) * t)),
        int(round(b1 + (b2 - b1) * t)),
    ))

def lighten(c, frac=.35):
    return _mix(c, '#ffffff', frac)


def darken(c, frac=.25):
    return _mix(c, '#000000', frac)


def dataset_color_map(
    labels,
    base_for_size=None,
    *,
    dm_color='#7f7f7f',
    street_lighten=.35,
    crime_darken=.25,
    street_kw='Street Buffer',
    crime_kw='Crime Point Buffer',
    direct_kw='Direct Mapping',
    default='#000000',
):
    base = {**_DEFAULT_BASE, **({str(k): v for k, v in (base_for_size or {}).items()})}
    cmap = {}
    for lab in labels:
        if direct_kw in lab:
            cmap[lab] = dm_color
            continue
        m = _SIZE_RE.search(lab)
        b = base.get(m.group(1) if m else None, default)
        if street_kw in lab:
            cmap[lab] = lighten(b, street_lighten)
        elif crime_kw in lab:
            cmap[lab] = darken(b, crime_darken)
        else:
            cmap[lab] = b
    return cmap

<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgba(62, 128, 234, 1);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ boxplots and stats ★ </h3>
</div>

In [ ]:
_SIZE_RE = re.compile(r"(\d+)m")


# choose which csvs you want to be plotted
def filter_csvs_by_size(csvs, keep_sizes={10, 15, 20, 25}, include_direct=True):
    out = []
    for p in csvs:
        name = p.name
        if include_direct and "direct_mapping" in name:
            out.append(p)
            continue
        m = _SIZE_RE.search(name)
        if m and int(m.group(1)) in keep_sizes:
            out.append(p)
    return out

def _sort_labels(labels):
    def key_fn(lab):
        if "Direct Mapping" in lab:
            return (-1, -1, -1)
        m = _SIZE_RE.search(lab)
        size = int(m.group(1)) if m else 10**9
        t = 0 if "Street Buffer" in lab else (1 if "Crime Point Buffer" in lab else 2)
        return (size, t, lab)
    return sorted(labels, key=key_fn)

# stacks boxplots, saves as interactive html file and image
def boxplot_day_night_pair_plotly(
    csvs,
    d_col, n_col,
    label_from_path,
    build_long,
    dataset_color_map,
    DAY_BASE=None,
    NIGHT_BASE=None,
    *,
    out_dir=Path("../crime/data/graphs/boxplots"),
    filename_stem,
    width,
    height,
    outlier_opacity
):
    out_dir.mkdir(parents=True, exist_ok=True)

    # map dataset
    csvs_dict = {label_from_path(p): str(p) for p in csvs}

    # build longs
    long_day = build_long(d_col, csvs_dict)
    long_night = build_long(n_col, csvs_dict)

    # sort categories
    cats_day = _sort_labels(list(pd.unique(long_day['dataset'])))
    cats_night = _sort_labels(list(pd.unique(long_night['dataset'])))

    def _coerce_base(b):
        return b if isinstance(b, dict) else None

    # get colormap for boxplots
    cmap_day = dataset_color_map(cats_day, base_for_size=_coerce_base(DAY_BASE))
    cmap_night = dataset_color_map(cats_night, base_for_size=_coerce_base(NIGHT_BASE))

    # update day figure display here
    fig_day = px.box(
        long_day,
        x="dataset", y="value",
        category_orders={"dataset": cats_day},
        color="dataset",
        color_discrete_map=cmap_day,
        points="outliers",
        title=f"Cross-blended Daytime Buffer Safety Score by Dataset",
        
    )

    # update night figure display here
    fig_night = px.box(
        long_night,
        x="dataset", y="value",
        category_orders={"dataset": cats_night},
        color="dataset",
        color_discrete_map=cmap_night,
        points="outliers",
        title=f"Cross-blended Nighttime Buffer Safety Score by Dataset"
    )

    # make outliers semi-transparent
    for tr in fig_day.data:
        tr.marker.opacity = outlier_opacity
    for tr in fig_night.data:
        tr.marker.opacity = outlier_opacity

    # stack vertically
    fig = make_subplots(rows=2, cols=1, subplot_titles=[fig_day.layout.title.text, fig_night.layout.title.text])

    for tr in fig_day.data:
        fig.add_trace(tr, row=1, col=1)
    for tr in fig_night.data:
        fig.add_trace(tr, row=2, col=1)

    def _range01(s): 
        return (0, 1) if (pd.api.types.is_numeric_dtype(s) and s.min() >= 0 and s.max() <= 1.01) else None
    r1 = _range01(long_day['value'])
    r2 = _range01(long_night['value'])
    if r1:
        fig.update_yaxes(range=r1, row=1, col=1)
    if r2:
        fig.update_yaxes(range=r2, row=2, col=1)

    # update title_font here to update the entire image/html display's title
    fig.update_layout(
        height=height, width=width,
        title_text="Day & Night Buffer Safety Score Boxplots",
        showlegend=False,
        margin=dict(t=80, b=60, l=40, r=20),
        title_font=dict(size=20),
        legend=dict(font=dict(size=14))
    )

    # update font_size in update_annotations here to increase each plots' title
    fig.update_xaxes(tickangle=-30, row=1, col=1, title_font=dict(size=20), tickfont=dict(size=20) )
    fig.update_xaxes(tickangle=-30, row=2, col=1, title_font=dict(size=20), tickfont=dict(size=20))
    fig.update_annotations(font_size=20)

    # save html
    safe_name = re.sub(r'[^a-z0-9._-]+', '_', filename_stem.lower())
    out_path = out_dir / f"{safe_name}.html"
    fig.write_html(str(out_path))
    print(f"[saved interactive plot] {out_path}")

    return fig

csvs_kept = filter_csvs_by_size(csvs, keep_sizes={10,15,20,25}, include_direct=True)

# update image size and other parameters for the entire plot here
fig = boxplot_day_night_pair_plotly(
    csvs=csvs_kept,
    d_col=d,
    n_col=n,
    label_from_path=label_from_path,
    build_long=build_long,
    dataset_color_map=dataset_color_map,
    DAY_BASE=DAY_BASE, 
    NIGHT_BASE=NIGHT_BASE,
    filename_stem=f"boxplots_{d}_{n}",
    width=1400,
    height=950,
    outlier_opacity=0.55
)


[saved interactive plot] ../crime/data/graphs/boxplots/boxplots_buffer_blended_day_buffer_blended_night.html


In [13]:
from typing import Callable, Dict, Iterable, Optional, Set, Tuple
_SIZE_RE = re.compile(r"(\d+)m")

# creates table of summary stats for each dataset, saves to csvs and as image
def summarize_scores(
    csvs: Iterable[Path],
    d_col: str,
    n_col: str,
    label_from_path: Callable[[Path], str],
    build_long: Callable[[str, Dict[str, str], Optional[str]], pd.DataFrame],
    keep_sizes: Optional[Set[int]] = None,
    include_direct: bool = True,
    round_to: int = 6,
    save_dir: Optional[Path] = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    def _filter(paths: Iterable[Path]) -> list[Path]:
        out: list[Path] = []
        for p in paths:
            name = p.name
            is_direct = "direct_mapping" in name
            if is_direct:
                if include_direct:
                    out.append(p)
                continue

            if keep_sizes is None:
                out.append(p)
            else:
                m = _SIZE_RE.search(name)
                if m and int(m.group(1)) in keep_sizes:
                    out.append(p)
        return out

    filtered = _filter(csvs)
    if not filtered:
        raise ValueError("No CSVs after filtering. Adjust keep_sizes/include_direct.")

    # build longs
    csvs_dict = {label_from_path(p): str(p) for p in filtered}
    long_day = build_long(d_col, csvs_dict)
    long_night = build_long(n_col, csvs_dict)

    # groupby/aggregate helper
    def _summ(long_df: pd.DataFrame) -> pd.DataFrame:
        g = long_df.groupby('dataset', dropna=False)['value']
        q1 = g.quantile(0.25).rename('Q1')
        q3 = g.quantile(0.75).rename('Q3')
        iqr = (q3 - q1).rename('IQR')

        agg = (
            g.agg(count='count', mean='mean', std='std', median='median', min='min', max='max')
             .reset_index()
             .merge(q1, on='dataset')
             .merge(q3, on='dataset')
             .merge(iqr, on='dataset')
        )

        # parse the dataset type
        parsed = agg.copy()
        parsed[['Type', 'Dist']] = parsed['dataset'].str.extract(r'(.+?) Buffer (\d+)m')
        parsed['Dist'] = pd.to_numeric(parsed['Dist'], errors='coerce')

        # sort by dataset
        order_map = {"Street": 0, "Crime Point": 1}
        parsed['type_order'] = parsed['Type'].map(order_map)
        is_direct = parsed['dataset'].str.contains('Direct Mapping', regex=False, na=False)
        parsed['direct_first'] = ~is_direct  # False (0) sorts first

        parsed = (
            parsed.sort_values(
                by=['direct_first', 'Dist', 'type_order', 'dataset'],
                ascending=[True, True, True, True]
            )
            .drop(columns=['type_order', 'direct_first'])
        )

        # round cells for presentation
        num_cols = ['mean', 'std', 'Q1', 'Q3', 'IQR', 'median', 'min', 'max']
        parsed[num_cols] = parsed[num_cols].round(round_to)
        return parsed

    stats_day = _summ(long_day)
    stats_night = _summ(long_night)

    # save to csv if wanted
    if save_dir is not None:
        save_dir.mkdir(parents=True, exist_ok=True)
        p_day = save_dir / f"summary_stats_day_{d_col}.csv"
        p_night = save_dir / f"summary_stats_night_{n_col}.csv"
        stats_day.to_csv(p_day, index=False)
        stats_night.to_csv(p_night, index=False)
        print(f"[saved] {p_day}")
        print(f"[saved] {p_night}")

    return stats_day, stats_night

def print_table(df: pd.DataFrame) -> None:
    """Aligned terminal print with consistent spacing/precision."""
    print(
        df.to_string(
            index=False,
            justify='right',
            col_space=12,
            float_format='{:>8.6f}'.format
        )
    )

# save table as image
def table_format_save(table: pd.DataFrame, name: str, out_dir: Path) -> None:
    from html2image import Html2Image

    table_html = table.to_html(index=False, border=0, classes="zebra")
    title_html = f"<h2 class='title'>{name.capitalize()} Summary</h2>"

    # zebra rows, sticky header, subtle borders, readable font
    full_html = f"""
    <html>
    <head>
      <meta charset="utf-8" />
      <style>
        :root {{
          --bg-even: #c7e8ff;
          --bg-odd:  #ffffff;
          --border:  #d1d5db;
          --header-bg: #111827;
          --header-fg: #ffffff;
          --text: #111827;
        }}
        * {{
          box-sizing: border-box;
        }}
        body {{
          background: white;
          margin: 0;
          padding: 12px;
          font-family: system-ui, -apple-system, Segoe UI, Roboto, Helvetica, Arial, "Apple Color Emoji", "Segoe UI Emoji";
          color: var(--text);
        }}
        .title {{
          font-size: 20px;
          font-weight: bold;
          margin-bottom: 12px;
          text-align: center;
        }}
        table {{
          border-collapse: separate;
          border-spacing: 0;
          width: 100%;
          overflow: hidden;
          border: 1px solid var(--border);
          border-radius: 12px;
        }}
        thead th {{
          position: sticky;
          top: 0;
          background: var(--header-bg);
          color: var(--header-fg);
          text-align: right;
          padding: 10px 12px;
          font-weight: 600;
          border-bottom: 1px solid var(--border);
          white-space: nowrap;
        }}
        tbody td {{
          padding: 8px 12px;
          border-bottom: 1px solid var(--border);
          text-align: right;
          white-space: nowrap;
        }}
        tbody tr:nth-child(odd)  {{ background: var(--bg-odd);  }}
        tbody tr:nth-child(even) {{ background: var(--bg-even); }}

        /* Rounded corners */
        thead th:first-child {{ text-align: left; border-top-left-radius: 12px; }}
        thead th:last-child  {{ border-top-right-radius: 12px; }}
        tbody tr:last-child td:first-child {{ border-bottom-left-radius: 12px; }}
        tbody tr:last-child td:last-child  {{ border-bottom-right-radius: 12px; }}

        /* Left-align the first column (dataset names) for readability */
        thead th:first-child, tbody td:first-child {{ text-align: left; }}
      </style>
    </head>
    <body>
      {title_html}
      {table_html}
    </body>
    </html>
    """

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # save screenshot, tweak size if needed
    hti = Html2Image(output_path=str(out_dir))
    hti.screenshot(
        html_str=full_html,
        save_as=f"{name}_summary_table.png",
        size=(1400, 800)
    )


keep = {10, 15, 20, 25, 40, 50}
stats_d, stats_n = summarize_scores(
    csvs=csvs,
    d_col=d,
    n_col=n,
    label_from_path=label_from_path,
    build_long=build_long,
    keep_sizes=keep,
    include_direct=True,
    round_to=6,
    save_dir=Path("../crime/data/graphs/stats")
)

print_table(stats_d)
print_table(stats_n)

# iamges with alternating row colors 
out_dir = Path("../crime/data/graphs/stats")
table_format_save(stats_d, "day", out_dir)
table_format_save(stats_n, "night", out_dir)


[saved] ../crime/data/graphs/stats/summary_stats_day_buffer_blended_day.csv
[saved] ../crime/data/graphs/stats/summary_stats_night_buffer_blended_night.csv
          dataset        count         mean          std       median          min          max           Q1           Q3          IQR         Type         Dist
   Direct Mapping        18972     0.988623     0.073429     1.000000     0.100000     1.000000     1.000000     1.000000     0.000000          NaN          NaN
Street Buffer 10m        18972     0.964844     0.129453     1.000000     0.100000     1.000000     0.993920     1.000000     0.006080       Street    10.000000
 Point Buffer 10m        18972     0.964848     0.129454     1.000000     0.100000     1.000000     0.993948     1.000000     0.006052        Point    10.000000
Street Buffer 15m        18972     0.896681     0.215346     0.989111     0.100000     1.000000     0.921025     1.000000     0.078975       Street    15.000000
 Point Buffer 15m        18972     0.89

119966 bytes written to file /Users/alice/Documents/GitHub/STARSWalkability/crime/data/graphs/stats/day_summary_table.png
119244 bytes written to file /Users/alice/Documents/GitHub/STARSWalkability/crime/data/graphs/stats/night_summary_table.png
